# Forward inference diagnostics

This notebook compares baseline OLS forward and discount-factor estimates with a spread-weighted fit. It inspects parity residuals and quote quality without applying production filters.

Saved quote-bearing outputs are intentionally omitted from the public notebook. Run locally with licensed data; see [notebook guide](README.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MATCHED_DATA_PATH = Path("../data/processed/spx_option_prices_matched_2025-08-29.csv")
FORWARD_DATA_PATH = Path("../data/processed/spx_forward_estimates_2025-08-29.csv")
GROUP_KEYS = ["security_id", "quote_date", "expiry_date"]

matched = pd.read_csv(MATCHED_DATA_PATH, parse_dates=["quote_date", "expiry_date"])
baseline = pd.read_csv(FORWARD_DATA_PATH, parse_dates=["quote_date", "expiry_date"])

matched.shape, baseline.shape

## Baseline question

The notebook asks whether wide or otherwise poor option quotes materially change the inferred forward $F$ and discount factor $D$. The matched panel contains one call-put pair per strike, while the baseline table contains one estimate per expiry.

The expected shapes are approximately $(7835, 21)$ for the matched panel and $(38, 9)$ for the baseline estimates. The baseline excludes the single expiry-day group, leaving 7,426 positive-DTE pairs across 38 expiries.

## Parity diagnostics

Put-call parity gives

$$
C_K-P_K=D(F-K)=DF-DK.
$$

For each expiry, the baseline OLS fit treats $C_K-P_K$ as a straight-line function of strike. Its slope estimates $-D$ and its intercept estimates $DF$. The diagnostics below measure how individual observations sit around that fitted line.

In [ ]:
diagnostics = matched[matched["time_to_expiry"] > 0].copy()
diagnostics["parity_value"] = diagnostics["call_mid"] - diagnostics["put_mid"]
diagnostics["combined_spread"] = (
    diagnostics["call_ask"] - diagnostics["call_bid"]
    + diagnostics["put_ask"] - diagnostics["put_bid"]
)

diagnostics = diagnostics.merge(
    baseline[GROUP_KEYS + ["forward", "discount_factor"]],
    on=GROUP_KEYS,
    how="left",
    validate="many_to_one",
)
diagnostics["fitted_parity"] = diagnostics["discount_factor"] * (
    diagnostics["forward"] - diagnostics["strike"]
)
diagnostics["parity_residual"] = (
    diagnostics["parity_value"] - diagnostics["fitted_parity"]
)

diagnostics[["parity_residual", "combined_spread"]].describe()

## Residual and spread measures

The parity value is $C_{\mathrm{mid}}-P_{\mathrm{mid}}$. The residual is

$$
\varepsilon_K=(C_K-P_K)-\hat D(\hat F-K).
$$

$\varepsilon_K=0$ means the pair lies on the fitted parity line; positive or negative values show the direction of the deviation. The combined spread is

$$
(C_{\mathrm{ask}}-C_{\mathrm{bid}})+(P_{\mathrm{ask}}-P_{\mathrm{bid}}),
$$

so it captures quote uncertainty from both legs. We would like residuals to remain fairly small and centred near zero. In the current sample, the residual standard deviation is about 0.71 and the median combined spread is 9.8 points.

In [ ]:
quote_quality = diagnostics.assign(
    zero_bid=(diagnostics["call_bid"].eq(0) | diagnostics["put_bid"].eq(0)),
    wide_spread=diagnostics["combined_spread"]
    >= diagnostics["combined_spread"].quantile(0.95),
)

quote_quality.groupby(["zero_bid", "wide_spread"])[
    ["parity_residual", "combined_spread"]
].agg(["count", "mean", "median"])

## Exploratory quote-quality buckets

The `zero_bid` flag identifies pairs where either leg has a zero bid. `wide_spread` identifies the widest 5% of combined spreads. This percentile is an exploratory label, not a production filtering rule.

Compare the mean and median residuals across the buckets. A noticeable shift for zero-bid or wide-spread quotes would suggest that quote quality has a systematic effect. We do not remove any observations in this notebook.

## Observed parity line

Before inspecting residuals, this plot shows the observed $C-P$ values against strike together with the fitted parity line. The points should broadly follow a downward-sloping straight line. Large curvature or clear deviations in the wings would indicate that the simple parity fit deserves closer investigation.

In [ ]:
selected_expiries = (
    diagnostics["expiry_date"].drop_duplicates().sort_values().iloc[[0, 10, -1]]
)
fig, axes = plt.subplots(
    len(selected_expiries), 1, figsize=(9, 4 * len(selected_expiries)), squeeze=False
)

for row, expiry in enumerate(selected_expiries):
    sample = diagnostics[diagnostics["expiry_date"] == expiry].sort_values("strike")
    axes[row, 0].scatter(sample["strike"], sample["parity_value"], s=8, label="Observed C-P")
    axes[row, 0].plot(sample["strike"], sample["fitted_parity"], color="black", label="OLS fitted parity")
    axes[row, 0].set_title(f"C-P and fitted parity: {expiry.date()}")
    axes[row, 0].set_xlabel("Strike")
    axes[row, 0].set_ylabel("C-P")
    axes[row, 0].legend()

plt.tight_layout()

In [ ]:
selected_expiries = (
    diagnostics["expiry_date"].drop_duplicates().sort_values().iloc[[0, 10, -1]]
)
fig, axes = plt.subplots(
    len(selected_expiries), 2, figsize=(12, 4 * len(selected_expiries))
)

for row, expiry in enumerate(selected_expiries):
    sample = diagnostics[diagnostics["expiry_date"] == expiry]

    axes[row, 0].scatter(sample["strike"], sample["parity_residual"], s=8)
    axes[row, 0].axhline(0, color="black", linewidth=0.8)
    axes[row, 0].set_title(f"Parity residual vs strike: {expiry.date()}")
    axes[row, 0].set_xlabel("Strike")
    axes[row, 0].set_ylabel("Residual")

    axes[row, 1].scatter(
        sample["combined_spread"], sample["parity_residual"], s=8
    )
    axes[row, 1].axhline(0, color="black", linewidth=0.8)
    axes[row, 1].set_title(f"Parity residual vs spread: {expiry.date()}")
    axes[row, 1].set_xlabel("Combined spread")
    axes[row, 1].set_ylabel("Residual")

plt.tight_layout()

## Residual plots

The left-hand plots ask whether parity error depends systematically on strike, especially in the wings. A cloud scattered around the zero line is the desired result. The right-hand plots ask whether wider quotes produce larger or more dispersed residuals; a funnel-shaped pattern would indicate a quote-quality effect.

These plots are diagnostic only. No zero-bid, wide-spread, or wing quotes are removed here.

## Spread-weighted comparison

The weighted fit gives tighter call-put pairs more influence. It uses objective weights proportional to

$$
w_K=\frac{1}{\text{combined spread}_K^2}.
$$

Equivalently, the least-squares calculation uses $1/\text{combined spread}$ as the square-root weight. For example, a spread of 5 receives four times the weight of a spread of 10. The purpose here is comparison: we are not adopting a production estimator yet.

In [ ]:
def weighted_forward_and_discount_factor(group):
    group = group[group["combined_spread"] > 0]
    strikes = group["strike"].to_numpy(float)
    parity = group["parity_value"].to_numpy(float)
    sqrt_weights = 1 / group["combined_spread"].to_numpy(float)

    design = np.column_stack([np.ones(len(strikes)), strikes])
    weighted_design = design * sqrt_weights[:, None]
    weighted_parity = parity * sqrt_weights

    intercept, slope = np.linalg.lstsq(
        weighted_design,
        weighted_parity,
        rcond=None,
    )[0]

    discount_factor = -slope
    forward = intercept / discount_factor
    return forward, discount_factor


weighted_results = []
for keys, group in diagnostics.groupby(GROUP_KEYS):
    forward, discount_factor = weighted_forward_and_discount_factor(group)
    weighted_results.append({
        "security_id": keys[0],
        "quote_date": keys[1],
        "expiry_date": keys[2],
        "weighted_forward": forward,
        "weighted_discount_factor": discount_factor,
    })

weighted = pd.DataFrame(weighted_results)
comparison = baseline.merge(
    weighted,
    on=GROUP_KEYS,
    validate="one_to_one",
)
comparison["forward_change"] = (
    comparison["weighted_forward"] - comparison["forward"]
)
comparison["discount_factor_change"] = (
    comparison["weighted_discount_factor"]
    - comparison["discount_factor"]
)

comparison[[
    "expiry_date",
    "forward",
    "weighted_forward",
    "forward_change",
    "discount_factor",
    "weighted_discount_factor",
    "discount_factor_change",
]].head()

## Interpreting the comparison

If the weighted and unweighted estimates are close, the baseline OLS result is stable despite quote-quality differences. In the current sample, the weighted forward is about 0.84 index points higher on average, while the discount factor is about 0.000344 lower. Those changes are small relative to the level of the estimates, so the baseline fit appears robust enough for the next stage of analysis.

In [ ]:
comparison[["forward_change", "discount_factor_change"]].describe()

## Normalized parity residuals

To compare midpoint residuals with the amount of quote uncertainty available for trading, define

$$
R_K=\frac{2|\varepsilon_K|}{\text{combined spread}_K}.
$$

The factor of 2 makes $R_K=1$ the approximate boundary at which the fitted parity value lies outside the executable parity interval. Values above 1 are the breaches we want to investigate. Plotting $R_K$ against strike shows whether they concentrate in particular strikes or maturities. The final check asks whether one common $F,D$ can satisfy every executable interval within each expiry.

In [ ]:
normalized = diagnostics.copy()
normalized["normalized_residual"] = (
    2 * normalized["parity_residual"].abs() / normalized["combined_spread"]
)

selected_expiries = (
    normalized["expiry_date"].drop_duplicates().sort_values().iloc[[0, 10, -1]]
)
fig, axes = plt.subplots(
    len(selected_expiries), 1, figsize=(9, 4 * len(selected_expiries)), squeeze=False
)

for row, expiry in enumerate(selected_expiries):
    sample = normalized[normalized["expiry_date"] == expiry].sort_values("strike")
    axes[row, 0].scatter(
        sample["strike"], sample["normalized_residual"], s=10
    )
    axes[row, 0].axhline(1, color="black", linestyle="--", linewidth=0.8)
    axes[row, 0].axvline(
        sample["forward"].iloc[0],
        color="tab:orange",
        linestyle="--",
        linewidth=0.8,
    )
    axes[row, 0].set_title(f"Normalized parity residual: {expiry.date()}")
    axes[row, 0].set_xlabel("Strike")
    axes[row, 0].set_ylabel("R")

plt.tight_layout()

(normalized["normalized_residual"] > 1).sum()

from scipy.optimize import linprog

common_fit_rows = []
for group_keys, group in normalized.groupby(GROUP_KEYS):
    lower = group["call_bid"].to_numpy() - group["put_ask"].to_numpy()
    upper = group["call_ask"].to_numpy() - group["put_bid"].to_numpy()
    strike = group["strike"].to_numpy()

    a_ub = np.vstack([
        np.column_stack([-np.ones(len(group)), strike]),
        np.column_stack([np.ones(len(group)), -strike]),
    ])
    b_ub = np.concatenate([-lower, upper])
    result = linprog(
        c=[0.0, 0.0],
        A_ub=a_ub,
        b_ub=b_ub,
        bounds=[(None, None), (1e-12, None)],
        method="highs",
    )

    row = dict(zip(GROUP_KEYS, group_keys))
    row["common_fit_exists"] = result.success
    row["forward"] = result.x[0] / result.x[1] if result.success else np.nan
    row["discount_factor"] = result.x[1] if result.success else np.nan
    common_fit_rows.append(row)

common_interval_check = pd.DataFrame(common_fit_rows)
common_interval_check